# WikiArt Style Classification — Test 8 (Max Accuracy Push)

This notebook pushes beyond Tests 6 and 7 with a longer accuracy-focused setup.

### What is stronger in Test 8
- Longer training schedule with more patience
- Lower fine-tune learning rates for late-stage stability
- Stronger EMA for better final averaging
- Safer eval split logic (fixes the `label` split issue)
- Separate output files so earlier tests are not overwritten

In [1]:
import copy
import math
import os
import random
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

try:
    import timm
except ImportError as exc:
    raise ImportError("This notebook requires timm. Install with: pip install timm") from exc


@dataclass
class TrainConfig:
    model_name: str = "vit_base_patch16_384.augreg_in21k_ft_in1k"
    image_size: int = 448
    batch_size: int = 6
    effective_batch_size: int = 72
    head_epochs: int = 6
    ft_epochs: int = 72
    patience: int = 20
    warmup_epochs: int = 5
    head_lr: float = 3.5e-4
    ft_backbone_lr: float = 3e-6
    ft_head_lr: float = 1.2e-5
    weight_decay: float = 1.5e-4
    mixup_alpha: float = 0.65
    cutmix_alpha: float = 1.0
    mix_probability: float = 0.9
    label_smoothing: float = 0.07
    ema_decay: float = 0.9999
    grad_clip_norm: float = 1.0
    tta: bool = True
    use_weighted_sampler: bool = True
    seed: int = 42
    num_workers: int = 0
    progress_every: int = 300
    fast_mode: bool = False


class WikiArtStyleDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, image_root: Path, transform=None):
        self.rows = rows.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int):
        row = self.rows.iloc[idx]
        image_path = self.image_root / row["relative_path"]
        label = int(row["label"])
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.9997):
        self.decay = decay
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model: nn.Module):
        msd = model.state_dict()
        for k, v in self.ema_model.state_dict().items():
            if k in msd:
                model_v = msd[k].detach()
                if torch.is_floating_point(v):
                    v.mul_(self.decay).add_(model_v, alpha=1.0 - self.decay)
                else:
                    v.copy_(model_v)


class SoftTargetCrossEntropyWithLabelSmoothing(nn.Module):
    def __init__(self, smoothing: float = 0.0):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n_classes = logits.size(1)
        if targets.dim() == 1:
            with torch.no_grad():
                true_dist = torch.zeros_like(logits)
                true_dist.fill_(self.smoothing / (n_classes - 1))
                true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
            targets = true_dist
        log_probs = torch.log_softmax(logits, dim=1)
        return torch.mean(torch.sum(-targets * log_probs, dim=1))


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def discover_paths(project_root: Path) -> Tuple[Path, Path, Path]:
    wikiart_dir = project_root / "datasets" / "Wikiart"
    train_csv = wikiart_dir / "style_train.csv"
    val_csv = wikiart_dir / "style_val.csv"
    if not train_csv.exists() or not val_csv.exists():
        raise FileNotFoundError("Could not find style_train.csv and style_val.csv in datasets/Wikiart")
    return wikiart_dir, train_csv, val_csv


def load_style_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=["relative_path", "label"])
    df["label"] = df["label"].astype(int)
    return df


def make_eval_split(val_df: pd.DataFrame, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    sampled_parts = []
    for _, grp in val_df.groupby("label", sort=False):
        if len(grp) > 1:
            sampled_parts.append(grp.sample(frac=0.5, random_state=seed))
        else:
            sampled_parts.append(grp)

    eval_test = pd.concat(sampled_parts, axis=0)
    eval_val = val_df.drop(index=eval_test.index)

    if len(eval_val) == 0:
        eval_val = val_df.sample(frac=0.5, random_state=seed)
        eval_test = val_df.drop(index=eval_val.index)

    eval_test = eval_test[["relative_path", "label"]].reset_index(drop=True)
    eval_val = eval_val[["relative_path", "label"]].reset_index(drop=True)
    return eval_val, eval_test


def filter_existing_rows(df: pd.DataFrame, image_root: Path, split_name: str) -> pd.DataFrame:
    keep_mask = df["relative_path"].map(lambda p: (image_root / p).exists())
    missing_count = int((~keep_mask).sum())
    if missing_count > 0:
        print(f"[{split_name}] Filtered out {missing_count} missing files.")
    return df.loc[keep_mask].reset_index(drop=True)


def build_transforms(image_size: int):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.55, 1.0), ratio=(0.75, 1.33)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=11),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.04),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
        transforms.RandomErasing(p=0.22, scale=(0.02, 0.18), ratio=(0.3, 3.3), value="random"),
    ])
    eval_tfms = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return train_tfms, eval_tfms


def create_weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    class_counts = np.bincount(labels)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[labels]
    sample_weights = np.sqrt(sample_weights)
    sample_weights = sample_weights / sample_weights.mean()
    return WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )


def mixup_cutmix(
    inputs: torch.Tensor,
    targets: torch.Tensor,
    alpha_mixup: float,
    alpha_cutmix: float,
    p: float,
    num_classes: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    if np.random.rand() > p:
        return inputs, torch.nn.functional.one_hot(targets, num_classes=num_classes).float()

    batch_size = inputs.size(0)
    indices = torch.randperm(batch_size, device=inputs.device)
    targets_onehot = torch.nn.functional.one_hot(targets, num_classes=num_classes).float()
    shuffled_targets = targets_onehot[indices]

    use_cutmix = alpha_cutmix > 0 and np.random.rand() < 0.5
    if use_cutmix:
        lam = np.random.beta(alpha_cutmix, alpha_cutmix)
        h, w = inputs.size(2), inputs.size(3)
        cut_rat = np.sqrt(1.0 - lam)
        cut_w = int(w * cut_rat)
        cut_h = int(h * cut_rat)

        cx = np.random.randint(w)
        cy = np.random.randint(h)

        x1 = np.clip(cx - cut_w // 2, 0, w)
        x2 = np.clip(cx + cut_w // 2, 0, w)
        y1 = np.clip(cy - cut_h // 2, 0, h)
        y2 = np.clip(cy + cut_h // 2, 0, h)

        mixed = inputs.clone()
        mixed[:, :, y1:y2, x1:x2] = inputs[indices, :, y1:y2, x1:x2]
        lam_adjusted = 1.0 - ((x2 - x1) * (y2 - y1) / (w * h))
        soft_targets = targets_onehot * lam_adjusted + shuffled_targets * (1.0 - lam_adjusted)
        return mixed, soft_targets

    if alpha_mixup <= 0:
        return inputs, targets_onehot

    lam = np.random.beta(alpha_mixup, alpha_mixup)
    mixed = lam * inputs + (1.0 - lam) * inputs[indices]
    soft_targets = lam * targets_onehot + (1.0 - lam) * shuffled_targets
    return mixed, soft_targets


def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, topk=(1, 5)) -> List[torch.Tensor]:
    with torch.no_grad():
        maxk = max(topk)
        _, pred = logits.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(targets.view(1, -1).expand_as(pred))
        out = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0)
            out.append(correct_k / targets.size(0))
        return out


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: Optional[torch.optim.Optimizer],
    device: torch.device,
    scaler: torch.amp.GradScaler,
    num_classes: int,
    cfg: TrainConfig,
    ema: Optional[ModelEMA],
    is_train: bool,
    accum_steps: int,
) -> Dict[str, float]:
    model.train(is_train)

    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total_samples = 0

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    autocast_enabled = device.type == "cuda"
    total_batches = len(loader)
    progress_every = max(1, int(getattr(cfg, "progress_every", 300)))

    for step, (images, targets) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=autocast_enabled):
            if is_train:
                images_mixed, soft_targets = mixup_cutmix(
                    images,
                    targets,
                    cfg.mixup_alpha,
                    cfg.cutmix_alpha,
                    cfg.mix_probability,
                    num_classes,
                )
                logits = model(images_mixed)
                loss = criterion(logits, soft_targets)
            else:
                logits = model(images)
                loss = criterion(logits, targets)

            loss = loss / accum_steps

        if is_train:
            scaler.scale(loss).backward()
            if step % accum_steps == 0 or step == total_batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if ema is not None:
                    ema.update(model)

        with torch.no_grad():
            top1, top5 = topk_accuracy(logits, targets, topk=(1, 5))

        bs = images.size(0)
        total_samples += bs
        total_loss += loss.item() * accum_steps * bs
        total_top1 += float(top1.item()) * bs
        total_top5 += float(top5.item()) * bs

        if is_train and (step % progress_every == 0 or step == total_batches):
            avg_loss = total_loss / max(1, total_samples)
            avg_top1 = total_top1 / max(1, total_samples)
            print(f"  [train] batch {step}/{total_batches} | avg_loss={avg_loss:.4f} | avg_top1={avg_top1:.4f}")

    return {
        "loss": total_loss / max(1, total_samples),
        "top1": total_top1 / max(1, total_samples),
        "top5": total_top5 / max(1, total_samples),
    }


@torch.no_grad()
def evaluate_with_tta(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    use_tta: bool,
) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total_samples = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        logits = model(images)
        if use_tta:
            logits_flip = model(torch.flip(images, dims=[3]))
            logits = (logits + logits_flip) / 2.0

        loss = criterion(logits, targets)
        top1, top5 = topk_accuracy(logits, targets, topk=(1, 5))

        bs = images.size(0)
        total_samples += bs
        total_loss += float(loss.item()) * bs
        total_top1 += float(top1.item()) * bs
        total_top5 += float(top5.item()) * bs

    return {
        "loss": total_loss / max(1, total_samples),
        "top1": total_top1 / max(1, total_samples),
        "top5": total_top5 / max(1, total_samples),
    }


def freeze_backbone(model: nn.Module, freeze: bool):
    for p in model.parameters():
        p.requires_grad = not freeze

    if hasattr(model, "head") and isinstance(model.head, nn.Module):
        for p in model.head.parameters():
            p.requires_grad = True

    if hasattr(model, "fc") and isinstance(model.fc, nn.Module):
        for p in model.fc.parameters():
            p.requires_grad = True


def get_trainable_parameters(model: nn.Module, backbone_lr: float, head_lr: float, weight_decay: float):
    head_params = []
    backbone_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(k in name for k in ["head", "fc", "classifier"]):
            head_params.append(p)
        else:
            backbone_params.append(p)

    param_groups = []
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": backbone_lr, "weight_decay": weight_decay})
    if head_params:
        param_groups.append({"params": head_params, "lr": head_lr, "weight_decay": weight_decay})
    return param_groups


def build_scheduler(optimizer: torch.optim.Optimizer, warmup_epochs: int, total_epochs: int):
    warmup_epochs = max(0, warmup_epochs)
    total_epochs = max(1, total_epochs)
    if warmup_epochs > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.12, total_iters=warmup_epochs)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_epochs - warmup_epochs))
        return torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_epochs])
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)


def pick_num_workers(explicit_workers: int) -> int:
    if explicit_workers >= 0:
        return explicit_workers
    if os.name == "nt":
        return 0
    return min(8, max(2, (os.cpu_count() or 4) // 2))


def fit(project_root: Path, cfg: TrainConfig):
    set_seed(cfg.seed)
    Image.MAX_IMAGE_PIXELS = None

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    wikiart_dir, train_csv, val_csv = discover_paths(project_root)

    train_df = load_style_csv(train_csv)
    val_df = load_style_csv(val_csv)

    train_df = filter_existing_rows(train_df, wikiart_dir, split_name="train")
    val_df = filter_existing_rows(val_df, wikiart_dir, split_name="val")

    if len(train_df) == 0 or len(val_df) == 0:
        raise RuntimeError("No usable data after filtering missing files.")

    eval_val_df, eval_test_df = make_eval_split(val_df, seed=cfg.seed)

    num_classes = int(max(train_df["label"].max(), val_df["label"].max()) + 1)

    train_tfms, eval_tfms = build_transforms(cfg.image_size)

    train_ds = WikiArtStyleDataset(train_df, wikiart_dir, transform=train_tfms)
    val_ds = WikiArtStyleDataset(eval_val_df, wikiart_dir, transform=eval_tfms)
    test_ds = WikiArtStyleDataset(eval_test_df, wikiart_dir, transform=eval_tfms)

    num_workers = pick_num_workers(cfg.num_workers)
    pin_memory = device.type == "cuda"

    sampler = None
    shuffle = True
    if cfg.use_weighted_sampler:
        sampler = create_weighted_sampler(train_df["label"].values)
        shuffle = False

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        sampler=sampler,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
    )

    if cfg.fast_mode:
        cfg.head_epochs = min(cfg.head_epochs, 1)
        cfg.ft_epochs = min(cfg.ft_epochs, 4)
        cfg.patience = min(cfg.patience, 2)

    print(f"Device: {device}")
    print(f"Model: {cfg.model_name}")
    print(f"Train/Val/Test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")
    print(f"Classes: {num_classes}")

    model = timm.create_model(cfg.model_name, pretrained=True, num_classes=num_classes)
    model.to(device)

    eval_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    train_criterion = SoftTargetCrossEntropyWithLabelSmoothing(smoothing=cfg.label_smoothing)

    grad_accum_steps = max(1, math.ceil(cfg.effective_batch_size / cfg.batch_size))

    scaler = torch.amp.GradScaler(device="cuda", enabled=device.type == "cuda")
    ema = ModelEMA(model, decay=cfg.ema_decay)

    history: List[Dict] = []
    best_val_top1 = -1.0
    best_epoch = -1
    patience_left = cfg.patience

    checkpoint_dir = project_root / "models"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / "wikiart_test8_best.pt"

    epoch_counter = 0

    for stage_name, stage_epochs, freeze_backbone_flag, lr_backbone, lr_head in [
        ("head-only", cfg.head_epochs, True, cfg.head_lr, cfg.head_lr),
        ("fine-tune", cfg.ft_epochs, False, cfg.ft_backbone_lr, cfg.ft_head_lr),
    ]:
        if stage_epochs <= 0:
            continue

        freeze_backbone(model, freeze=freeze_backbone_flag)
        param_groups = get_trainable_parameters(
            model,
            backbone_lr=lr_backbone,
            head_lr=lr_head,
            weight_decay=cfg.weight_decay,
        )
        optimizer = torch.optim.AdamW(param_groups)
        scheduler = build_scheduler(
            optimizer,
            warmup_epochs=min(cfg.warmup_epochs, stage_epochs),
            total_epochs=stage_epochs,
        )

        for _ in range(stage_epochs):
            epoch_counter += 1

            train_metrics = run_epoch(
                model=model,
                loader=train_loader,
                criterion=train_criterion,
                optimizer=optimizer,
                device=device,
                scaler=scaler,
                num_classes=num_classes,
                cfg=cfg,
                ema=ema,
                is_train=True,
                accum_steps=grad_accum_steps,
            )

            eval_model = ema.ema_model if ema is not None else model
            val_metrics = evaluate_with_tta(
                model=eval_model,
                loader=val_loader,
                criterion=eval_criterion,
                device=device,
                use_tta=cfg.tta,
            )

            scheduler.step()

            current_lr = optimizer.param_groups[0]["lr"]
            row = {
                "epoch": epoch_counter,
                "stage": stage_name,
                "lr": current_lr,
                "train_loss": train_metrics["loss"],
                "train_top1": train_metrics["top1"],
                "train_top5": train_metrics["top5"],
                "val_loss": val_metrics["loss"],
                "val_top1": val_metrics["top1"],
                "val_top5": val_metrics["top5"],
            }
            history.append(row)

            print(
                f"Epoch {epoch_counter:02d} | {stage_name:9s} | "
                f"lr={current_lr:.2e} | "
                f"train_loss={row['train_loss']:.4f}, train_acc={row['train_top1']:.4f} | "
                f"val_loss={row['val_loss']:.4f}, val_acc={row['val_top1']:.4f}, val_top5={row['val_top5']:.4f}"
            )

            improved = row["val_top1"] > best_val_top1
            if improved:
                best_val_top1 = row["val_top1"]
                best_epoch = epoch_counter
                patience_left = cfg.patience
                checkpoint = {
                    "model_state": copy.deepcopy(eval_model.state_dict()),
                    "config": asdict(cfg),
                    "best_val_top1": best_val_top1,
                    "best_epoch": best_epoch,
                    "num_classes": num_classes,
                    "model_name": cfg.model_name,
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                }
                torch.save(checkpoint, checkpoint_path)
            else:
                patience_left -= 1

            if patience_left <= 0:
                print(f"Early stopping at epoch {epoch_counter} (best epoch: {best_epoch}).")
                break

        if patience_left <= 0:
            break

    if checkpoint_path.exists():
        state = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(state["model_state"])

    final_val_metrics = evaluate_with_tta(model, val_loader, eval_criterion, device=device, use_tta=cfg.tta)
    final_test_metrics = evaluate_with_tta(model, test_loader, eval_criterion, device=device, use_tta=cfg.tta)

    print("\nFinal evaluation")
    print(
        f"Validation -> loss: {final_val_metrics['loss']:.4f}, "
        f"top1: {final_val_metrics['top1']:.4f}, top5: {final_val_metrics['top5']:.4f}"
    )
    print(
        f"Test       -> loss: {final_test_metrics['loss']:.4f}, "
        f"top1: {final_test_metrics['top1']:.4f}, top5: {final_test_metrics['top5']:.4f}"
    )
    print(f"Best checkpoint: {checkpoint_path}")
    print(f"Best epoch: {best_epoch}")
    print(f"Best val_top1: {best_val_top1:.4f}")

    history_df = pd.DataFrame(history)
    results_dir = project_root / "models" / "results"
    results_dir.mkdir(parents=True, exist_ok=True)

    history_csv = results_dir / "wikiart_test8_history.csv"
    history_df.to_csv(history_csv, index=False)

    summary_row = {
        "notebook": "wikiart_style_classification_test8_max_accuracy.ipynb",
        "exists": True,
        "model_name": cfg.model_name,
        "val_loss": final_val_metrics["loss"],
        "val_top1": final_val_metrics["top1"],
        "val_top5": final_val_metrics["top5"],
        "test_loss": final_test_metrics["loss"],
        "test_top1": final_test_metrics["top1"],
        "test_top5": final_test_metrics["top5"],
        "best_val_top1": best_val_top1,
        "best_epoch": best_epoch,
        "experiment": "test8",
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    summary_csv = results_dir / "wikiart_tests_1_to_8_summary.csv"
    prior_summary_csv = results_dir / "wikiart_tests_1_to_7_summary.csv"

    summary_rows = []
    if prior_summary_csv.exists():
        prior_df = pd.read_csv(prior_summary_csv)
        summary_rows.extend(prior_df.to_dict(orient="records"))

    summary_rows = [r for r in summary_rows if str(r.get("experiment", "")).lower() != "test8"]
    summary_rows.append(summary_row)

    summary_df = pd.DataFrame(summary_rows)
    if "experiment" in summary_df.columns:
        summary_df = summary_df.sort_values("experiment").reset_index(drop=True)
    summary_df.to_csv(summary_csv, index=False)

    print(f"Saved history: {history_csv}")
    print(f"Saved summary: {summary_csv}")

    return history_df, final_val_metrics, final_test_metrics

In [2]:
# Progress override: print epoch start + progress every N batches

def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: Optional[torch.optim.Optimizer],
    device: torch.device,
    scaler: torch.amp.GradScaler,
    num_classes: int,
    cfg: TrainConfig,
    ema: Optional[ModelEMA],
    is_train: bool,
    accum_steps: int,
) -> Dict[str, float]:
    model.train(is_train)

    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total_samples = 0

    if is_train:
        optimizer.zero_grad(set_to_none=True)
        run_epoch.epoch_counter = getattr(run_epoch, "epoch_counter", 0) + 1
        print(f"\n--- Starting epoch {run_epoch.epoch_counter} ---")

    autocast_enabled = device.type == "cuda"
    total_batches = len(loader)
    progress_every = max(1, int(getattr(cfg, "progress_every", 300)))

    for step, (images, targets) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=autocast_enabled):
            if is_train:
                images_mixed, soft_targets = mixup_cutmix(
                    images,
                    targets,
                    cfg.mixup_alpha,
                    cfg.cutmix_alpha,
                    cfg.mix_probability,
                    num_classes,
                )
                logits = model(images_mixed)
                loss = criterion(logits, soft_targets)
            else:
                logits = model(images)
                loss = criterion(logits, targets)

            loss = loss / accum_steps

        if is_train:
            scaler.scale(loss).backward()
            if step % accum_steps == 0 or step == total_batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if ema is not None:
                    ema.update(model)

        with torch.no_grad():
            top1, top5 = topk_accuracy(logits, targets, topk=(1, 5))

        bs = images.size(0)
        total_samples += bs
        total_loss += loss.item() * accum_steps * bs
        total_top1 += float(top1.item()) * bs
        total_top5 += float(top5.item()) * bs

        if is_train and (step % progress_every == 0 or step == total_batches):
            avg_loss = total_loss / max(1, total_samples)
            avg_top1 = total_top1 / max(1, total_samples)
            print(
                f"  [train] epoch {run_epoch.epoch_counter} | "
                f"batch {step}/{total_batches} | avg_loss={avg_loss:.4f} | avg_top1={avg_top1:.4f}"
            )

    return {
        "loss": total_loss / max(1, total_samples),
        "top1": total_top1 / max(1, total_samples),
        "top5": total_top5 / max(1, total_samples),
    }


# Wrap fit so epoch counter resets on each fresh training run
_original_fit = fit

def fit(project_root: Path, cfg: TrainConfig):
    run_epoch.epoch_counter = 0
    return _original_fit(project_root, cfg)

print("Progress override active: epoch starts + every 300 batches.")

Progress override active: epoch starts + every 300 batches.


In [3]:
cfg = TrainConfig(
    model_name="vit_base_patch16_384.augreg_in21k_ft_in1k",
    image_size=384,
    batch_size=6,
    effective_batch_size=72,
    head_epochs=6,
    ft_epochs=72,
    patience=20,
    warmup_epochs=5,
    head_lr=3.5e-4,
    ft_backbone_lr=3e-6,
    ft_head_lr=1.2e-5,
    weight_decay=1.5e-4,
    mixup_alpha=0.65,
    cutmix_alpha=1.0,
    mix_probability=0.9,
    label_smoothing=0.07,
    ema_decay=0.9999,
    grad_clip_norm=1.0,
    tta=True,
    use_weighted_sampler=True,
    seed=42,
    num_workers=0,
    progress_every=300,
)

# Kept at 384 because this pretrained model variant is fixed to 384 input.

In [4]:
history_df, final_val, final_test = fit(Path.cwd(), cfg)

[train] Filtered out 2 missing files.
Device: cuda
Model: vit_base_patch16_384.augreg_in21k_ft_in1k
Train/Val/Test sizes: 57023/12213/12208
Classes: 27

--- Starting epoch 1 ---
  [train] epoch 1 | batch 300/9503 | avg_loss=4.3446 | avg_top1=0.0372
  [train] epoch 1 | batch 600/9503 | avg_loss=4.2082 | avg_top1=0.0456
  [train] epoch 1 | batch 900/9503 | avg_loss=4.0910 | avg_top1=0.0543
  [train] epoch 1 | batch 1200/9503 | avg_loss=4.0067 | avg_top1=0.0576
  [train] epoch 1 | batch 1500/9503 | avg_loss=3.9331 | avg_top1=0.0597
  [train] epoch 1 | batch 1800/9503 | avg_loss=3.8697 | avg_top1=0.0636
  [train] epoch 1 | batch 2100/9503 | avg_loss=3.8225 | avg_top1=0.0669
  [train] epoch 1 | batch 2400/9503 | avg_loss=3.7800 | avg_top1=0.0702
  [train] epoch 1 | batch 2700/9503 | avg_loss=3.7415 | avg_top1=0.0741
  [train] epoch 1 | batch 3000/9503 | avg_loss=3.6976 | avg_top1=0.0777
  [train] epoch 1 | batch 3300/9503 | avg_loss=3.6609 | avg_top1=0.0808
  [train] epoch 1 | batch 3600/95

In [5]:
print("History rows:", len(history_df))
if len(history_df) > 0:
    best_idx = history_df["val_top1"].idxmax()
    print("\nBest validation row:")
    display(history_df.loc[[best_idx]])

print("\nFinal Validation:", final_val)
print("Final Test:", final_test)

results_dir = Path.cwd() / "models" / "results"
print("\nSaved files:")
print("-", results_dir / "wikiart_test8_history.csv")
print("-", results_dir / "wikiart_tests_1_to_8_summary.csv")
print("-", Path.cwd() / "models" / "wikiart_test8_best.pt")

summary_path = results_dir / "wikiart_tests_1_to_8_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path).tail(3))

History rows: 78

Best validation row:


,epoch,stage,lr,train_loss,train_top1,train_top5,val_loss,val_top1,val_top5
73,74,fine-tune,2.630619e-08,0.587713,0.716896,0.956312,1.473577,0.76656,0.974863



Final Validation: {'loss': 1.4735772922155577, 'top1': 0.7665602425280322, 'top5': 0.9748628562335973}
Final Test: {'loss': 1.5120000836010998, 'top1': 0.7594200714807445, 'top5': 0.9692824437804178}

Saved files:
- c:\Users\Thijs\Desktop\Ai Art Critic\models\results\wikiart_test8_history.csv
- c:\Users\Thijs\Desktop\Ai Art Critic\models\results\wikiart_tests_1_to_8_summary.csv
- c:\Users\Thijs\Desktop\Ai Art Critic\models\wikiart_test8_best.pt


,notebook,exists,model_name,val_loss,val_top1,val_top5,test_loss,test_top1,test_top5,best_val_top1,best_epoch,experiment,saved_at_utc
5,wikiart_style_classification_test6_max_accurac...,True,vit_base_patch16_384.augreg_in21k_ft_in1k,1.888643,0.758536,0.975845,1.926972,0.748689,0.972068,0.758536,24.0,test6,2026-03-16T11:19:13.132687+00:00
6,wikiart_style_classification_test7_max_accurac...,True,vit_base_patch16_384.augreg_in21k_ft_in1k,1.527187,0.765250,0.975845,1.568279,0.758355,0.972149,0.765250,45.0,test7,2026-03-19T00:50:52.171441+00:00
7,wikiart_style_classification_test8_max_accurac...,True,vit_base_patch16_384.augreg_in21k_ft_in1k,1.473577,0.766560,0.974863,1.512000,0.759420,0.969282,0.766560,74.0,test8,2026-03-22T05:38:36.051056+00:00
